# ATIS SLU & S&P 500 Experiments

This notebook contains end-to-end experiments for:

- Question 1: RNN-based SLU on ATIS (Slot Filling + Intent Detection)
- Question 2: Recursive time-series forecasting for S&P 500 (Baseline MLP, CNN+LSTM, GRU, TCN)

Follow the sections below to reproduce dataset download, preprocessing, model training and evaluation. Use the scripts under `scripts/` for full training runs.

## Section 1 — Environment Setup & Imports

Install required packages (run in a cell if not installed) and import libraries. Detect device (CUDA / MPS / CPU) and set seeds for reproducibility.

```python
# Install (run once)
# !pip install -r ../requirements.txt

import os
import random
import numpy as np
import torch
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)
```

In [ ]:
# Imports used across the notebook
import json
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.data.preprocess import load_atis_examples, build_vocab, build_label_vocab, ATISDataset, collate_fn
from src.models.baseline import BiRNNSlotFiller, BiLSTMJoint
from src.utils.metrics import slot_f1, slot_classification_report, intent_accuracy

sns.set(style="whitegrid")
print("imports ok")

## Section 2 — Download & Load ATIS Dataset

We use `kagglehub` helper to download the `siddhadev/atis-dataset-clean` dataset and then parse it with our loader.

```python
import kagglehub
path = kagglehub.dataset_download("siddhadev/atis-dataset-clean")
print("Downloaded to:", path)

# load
examples = load_atis_examples(path)
len(examples)
```

In [ ]:
# Try load processed if exists else use raw
proc_dir = Path("../data/processed").resolve()
if proc_dir.exists():
    train = json.load(open(proc_dir / "train.json", "r", encoding="utf8"))
    dev = json.load(open(proc_dir / "dev.json", "r", encoding="utf8"))
    test = json.load(open(proc_dir / "test.json", "r", encoding="utf8"))
    print(f"Loaded processed: train={len(train)} dev={len(dev)} test={len(test)}")
else:
    print("Processed data not found. Run preprocessing script: python -m src.data.preprocess --input data/raw --out data/processed")